# Siamese Network for One-Shot Facial Recognition

This notebook implements a Siamese neural network for facial identity verification. The model learns a similarity metric between image pairs, enabling one-shot recognition without retraining on new identities.

## 1. Dependencies and Environment Setup

In [1]:
import cv2
import os
import numpy as np
from matplotlib import pyplot as plt
import uuid
import tensorflow as tf
import keras
from keras.models import Model
from keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
from keras.metrics import Precision, Recall

## 2. Data Directory Configuration

Three disjoint image sets are defined: **anchor** (reference face), **positive** (same identity as anchor), and **negative** (different identity). This triplet structure is required for contrastive learning.

In [2]:
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

## 3. Image Acquisition

Frames are captured via webcam and written to disk on keypress. Press **** to save an anchor sample, **** to save a positive sample, **** to quit.

In [3]:
cap = cv2.VideoCapture(0)

In [4]:
while cap.isOpened():
    ret, frame = cap.read()

    frame = cv2.resize(frame, (250, 250))

    key = cv2.waitKey(1) & 0xFF

    if key == ord('a'):
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    if key == ord('p'):
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    cv2.imshow('image collection', frame)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 4. Dataset Construction

Image paths are loaded into  pipelines. Anchor-positive pairs receive label 1; anchor-negative pairs receive label 0. The two subsets are concatenated into a unified binary-classification dataset.

In [4]:
anchor   = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg').take(3000)
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg').take(3000)
negative = tf.data.Dataset.list_files(NEG_PATH + '/*/*.jpg').take(3000) 
dir_test = anchor.as_numpy_iterator()

## 5. Image Preprocessing

Each image is decoded from JPEG, resized to 100×100 pixels, and normalised to [0, 1]. Normalisation stabilises gradient flow during training.

In [5]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(byte_img)
    
    img = tf.image.resize(img, (100,100))
    img = img / 255.0

    return img

## 6. Paired Dataset Assembly

Anchor-image pairs are zipped with their binary labels and concatenated.  applies the preprocessing pipeline element-wise to both images in each pair.

In [6]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(3000))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(3000))))
data = positives.concatenate(negatives)

def preprocess_twin(input_img, validation_img, label):
    return(preprocess(input_img), preprocess(validation_img), label)

## 7. Pipeline Optimisation and Train/Test Split

The dataset is cached in memory, shuffled, and split 70/30 into training and test partitions. Batching (size 16) and prefetching overlap I/O with compute, reducing GPU starvation.

In [7]:
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=10000)

train_data = data.take(round(len(data)*.7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

test_data = data.skip(round(len(data)*.7))
test_data = test_data.take(round(len(data)*.3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)

## 8. Embedding Network Architecture

A convolutional feature extractor maps each 100×100×3 input to a 4096-dimensional embedding vector. Four successive Conv→MaxPool blocks progressively capture local to global spatial features. Sigmoid activation on the final dense layer constrains embeddings to the unit hypercube.

In [8]:
def make_embedding(): 
    inp = Input(shape=(100,100,3), name='input_image')
    
    c1 = Conv2D(64, (10,10), activation='relu')(inp)
    m1 = MaxPooling2D((2,2), padding='same')(c1)
    
    c2 = Conv2D(128, (7,7), activation='relu')(m1)
    m2 = MaxPooling2D((2,2), padding='same')(c2)

    c3 = Conv2D(128, (4,4), activation='relu')(m2)
    m3 = MaxPooling2D((2,2), padding='same')(c3)

    c4 = Conv2D(256, (4,4), activation='relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)
    
    return Model(inputs=[inp], outputs=d1, name='embedding') 


In [9]:
embedding = make_embedding()

## 9. L1 Distance Layer

The L1Dist layer computes the element-wise absolute difference between two embedding vectors. L1 distance serves as the similarity metric; smaller values indicate greater similarity. Implemented as a custom Keras layer to enable end-to-end gradient computation.

In [10]:
class L1Dist(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)   

    def call(self, inputs):
        input_embedding, validation_embedding = inputs
        if isinstance(input_embedding, (list, tuple)):
            input_embedding = input_embedding[0]
        if isinstance(validation_embedding, (list, tuple)):
            validation_embedding = validation_embedding[0]
        return tf.math.abs(input_embedding - validation_embedding)

    def compute_output_shape(self, input_shape):
        return input_shape[0]       

In [11]:
l1 = L1Dist()

## 10. Siamese Network Assembly

Both input branches share the same embedding network (weight-tied). Their L1 distance is passed through a single sigmoid neuron, producing a scalar match probability ∈ (0, 1). This architecture enforces metric symmetry by design.

In [12]:
def make_siamese_model(): 
    
    input_image = Input(name='input_img', shape=(100,100,3))
    
    validation_image = Input(name='validation_img', shape=(100,100,3))
    
    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer([embedding(input_image), embedding(validation_image)])
    
    classifier = Dense(1, activation='sigmoid')(distances)
    
    return Model(inputs=[input_image, validation_image], outputs=classifier, name='SiameseNetwork')


In [13]:
siamese_model = make_siamese_model()

## 11. Loss Function and Optimiser

Binary cross-entropy loss measures divergence between predicted match probability and ground-truth label. Adam (lr=1e-4) provides adaptive per-parameter learning rates, suitable for sparse gradients in deep networks.

In [14]:
binary_cross_loss = tf.losses.BinaryCrossentropy()
opt = tf.keras.optimizers.Adam(1e-4)

## 12. Training Checkpoints

TensorFlow checkpoints serialise optimiser state and model weights at regular intervals, enabling training resumption without loss of progress.

In [15]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_model)

## 13. Training Step

A single gradient update step is compiled into a TensorFlow graph via , eliminating Python interpreter overhead. Forward pass, loss computation, and backpropagation are enclosed within a  context for automatic differentiation.

In [16]:
@tf.function
def train_step(batch):
    
    with tf.GradientTape() as tape:
        X = batch[:2]
        y = batch[2]
        
        yhat = siamese_model(X, training=True)
        loss = binary_cross_loss(y, yhat)
    tf.print(loss)
        
    grad = tape.gradient(loss, siamese_model.trainable_variables)
    
    opt.apply_gradients(zip(grad, siamese_model.trainable_variables))
        
    return loss

## 14. Training Loop

The outer loop iterates over epochs; the inner loop processes each mini-batch. Precision and Recall are computed per epoch to monitor class-level performance beyond scalar loss. Checkpoints are saved every 10 epochs.

In [17]:
def train(data, EPOCHS):
    for epoch in range(1, EPOCHS+1):
        print('\n Epoch {}/{}'.format(epoch, EPOCHS))  # \n must be typed, not a real newline
        progbar = tf.keras.utils.Progbar(len(data))
        
        r = Recall()
        p = Precision()
        
        for idx, batch in enumerate(data):
            loss = train_step(batch)
            yhat = siamese_model(batch[:2], training=False)
            r.update_state(batch[2], yhat)
            p.update_state(batch[2], yhat)
            progbar.update(idx+1)
        print(loss.numpy(), r.result().numpy(), p.result().numpy())
        
        if epoch % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

In [19]:
EPOCHS = 50

In [20]:
train(train_data, EPOCHS)


 Epoch 1/50
0.69312
 1/28 ━━━━━━━━━━━━━━━━━━━━ 1:48 4s/step0.687945604
 2/28 ━━━━━━━━━━━━━━━━━━━━ 2:13 5s/step0.688875
 3/28 ━━━━━━━━━━━━━━━━━━━━ 1:43 4s/step0.666002154
 4/28 ━━━━━━━━━━━━━━━━━━━━ 1:24 4s/step0.629742563
 5/28 ━━━━━━━━━━━━━━━━━━━━ 1:14 3s/step0.614862144
 6/28 ━━━━━━━━━━━━━━━━━━━━ 1:09 3s/step0.647507906
 7/28 ━━━━━━━━━━━━━━━━━━━━ 1:14 4s/step0.717410445


KeyboardInterrupt: 

## 15. Inference and Evaluation

A held-out test batch is extracted and passed through the trained model. Predictions are thresholded at 0.5 to produce binary classifications. Precision and Recall are aggregated across all test batches to assess generalisation.

In [18]:
test_input, test_val, y_true = test_data.as_numpy_iterator().next()

In [19]:
y_hat = siamese_model.predict([test_input, test_val])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 671ms/step


In [20]:
[1 if prediction > 0.5 else 0 for prediction in y_hat ]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [21]:
y_true

array([1., 0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 0., 1., 0., 0., 0.],
      dtype=float32)

In [22]:
m = Recall()

m.update_state(y_true, y_hat)

m.result().numpy()

1.0

In [23]:
r = Recall()
p = Precision()

for test_input, test_val, y_true in test_data.as_numpy_iterator():
    yhat = siamese_model.predict([test_input, test_val])
    r.update_state(y_true, yhat)
    p.update_state(y_true,yhat) 

print(r.result().numpy(), p.result().numpy())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 879ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 907ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 846ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 840ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 783ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 656ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 610ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 652ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 637ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 571ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 689ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 633ms/step
0.9868421 0.40106952


2026-05-04 04:24:26.562864: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 16. Model Persistence

The trained Siamese network is serialised to the Keras native format (), preserving architecture, weights, and training configuration. The custom  layer is registered at load time via .

In [24]:
siamese_model.save('my_model.keras')

In [25]:
L1Dist

__main__.L1Dist

In [26]:
siamese_model = tf.keras.models.load_model('my_model.keras', 
                                   custom_objects={'L1Dist':L1Dist, 'BinaryCrossentropy':tf.losses.BinaryCrossentropy})

In [27]:
siamese_model.predict([test_input, test_val])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 822ms/step


array([[0.5026105 ],
       [0.50405633],
       [0.5022646 ],
       [0.5003229 ],
       [0.5017315 ],
       [0.5027262 ],
       [0.50179803],
       [0.50246066],
       [0.50047153],
       [0.50205094],
       [0.50223625],
       [0.50296265]], dtype=float32)

## 17. Verification Application

The deployed verification system compares a live webcam capture against a set of enrolled reference images. A face is verified if the proportion of reference images exceeding the detection threshold surpasses the verification threshold — a two-stage decision boundary that reduces both false accepts and false rejects.

In [28]:
# Create required directories if they don't exist
os.makedirs(os.path.join('application_data', 'verification_images'), exist_ok=True)
os.makedirs(os.path.join('application_data', 'input_image'), exist_ok=True)

print(os.listdir(os.path.join('application_data', 'verification_images')))
print(os.path.join('application_data', 'input_image', 'input_image.jpg'))

for image in os.listdir(os.path.join('application_data', 'verification_images')):
    validation_img = os.path.join('application_data', 'verification_images', image)
    print(validation_img)

['9a382f9e-4766-11f1-a12c-ce5d26bfadff.jpg', '7713e49a-4766-11f1-a12c-ce5d26bfadff.jpg', 'dde431c0-4766-11f1-a12c-ce5d26bfadff.jpg', '991ee382-4766-11f1-a12c-ce5d26bfadff.jpg', 'a747fb9c-4766-11f1-a12c-ce5d26bfadff.jpg', '80e8f5dc-4766-11f1-a12c-ce5d26bfadff.jpg', '6bc9ac80-4763-11f1-b792-ce5d26bfadff.jpg', '940f27aa-4791-11f1-a756-ce5d26bfadff.jpg', '6d9f1c02-4763-11f1-b792-ce5d26bfadff.jpg', '93a92f18-4791-11f1-a756-ce5d26bfadff.jpg', '9d26a4ce-4766-11f1-a12c-ce5d26bfadff.jpg', '8f8b17e8-4791-11f1-a756-ce5d26bfadff.jpg', '01de4a7c-4742-11f1-8344-ce5d26bfadff.jpg', '946a254c-4791-11f1-a756-ce5d26bfadff.jpg', '91d4fc44-4791-11f1-a756-ce5d26bfadff.jpg', '90b2bb94-4791-11f1-a756-ce5d26bfadff.jpg', '9a841a9e-4766-11f1-a12c-ce5d26bfadff.jpg', 'dd7215e0-4766-11f1-a12c-ce5d26bfadff.jpg', 'cf4b8dde-4766-11f1-a12c-ce5d26bfadff.jpg', '94a80146-4791-11f1-a756-ce5d26bfadff.jpg', '9338c4c6-4791-11f1-a756-ce5d26bfadff.jpg', '9187d2de-4791-11f1-a756-ce5d26bfadff.jpg', '92623776-4791-11f1-a756-ce5d26

In [29]:
import shutil

# Copy positive images into verification_images folder
pos_images = os.listdir(POS_PATH)[:200]  # 50 reference images is enough

for img in pos_images:
    shutil.copy(
        os.path.join(POS_PATH, img),
        os.path.join('application_data', 'verification_images', img)
    )

print(f"Copied {len(pos_images)} images into verification_images/")
print(os.listdir(os.path.join('application_data', 'verification_images'))[:5])

Copied 200 images into verification_images/
['9a382f9e-4766-11f1-a12c-ce5d26bfadff.jpg', '7713e49a-4766-11f1-a12c-ce5d26bfadff.jpg', 'dde431c0-4766-11f1-a12c-ce5d26bfadff.jpg', '991ee382-4766-11f1-a12c-ce5d26bfadff.jpg', 'a747fb9c-4766-11f1-a12c-ce5d26bfadff.jpg']


In [37]:
def verify(model, detection_threshold, verification_threshold):
    # Filter to only valid image files — excludes .gitkeep and other non-images
    verification_images = [
        f for f in os.listdir(os.path.join('application_data', 'verification_images'))
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if not verification_images:
        return np.array([]), False

    results = []
    for image in verification_images:
        input_img = preprocess(os.path.join('application_data', 'input_image', 'input_image.jpg'))
        validation_img = preprocess(os.path.join('application_data', 'verification_images', image))

        result = model.predict(
            [np.expand_dims(input_img, axis=0), np.expand_dims(validation_img, axis=0)],
            verbose=0
        )

        results.append(result[0][0])

    results = np.array(results)

    detection = np.sum(results > detection_threshold)
    verification = detection / len(verification_images)
    verified = verification > verification_threshold

    return results, verified

In [38]:
def preprocess(file_path):
    # Guard: ensure file exists and is non-empty
    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        raise ValueError(f"Image file is missing or empty: {file_path}")

    byte_img = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(byte_img)
    img = tf.image.resize(img, (100, 100))
    img = img / 255.0
    return img

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None:          # ← guard against bad frames
        continue

    frame = cv2.resize(frame, (250, 250))
    cv2.imshow('Verification', frame)

    key = cv2.waitKey(10) & 0xFF

    if key == ord('v'):
        # Ensure the directory exists before writing
        os.makedirs(os.path.join('application_data', 'input_image'), exist_ok=True)

        save_path = os.path.join('application_data', 'input_image', 'input_image.jpg')
        success = cv2.imwrite(save_path, frame)   # ← check return value

        if not success:
            print(f"ERROR: cv2.imwrite failed. Path: {save_path}")
            print(f"Frame shape: {frame.shape}, dtype: {frame.dtype}")
        else:
            # Only call verify if the file was actually written
            results, verified = verify(siamese_model, 0.5, 0.5)
            print(f"Scores: min={results.min():.3f} max={results.max():.3f} mean={results.mean():.3f}")
            print(f"Verified: {verified}")

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Scores: min=0.500 max=0.503 mean=0.502
Verified: True
Scores: min=0.500 max=0.504 mean=0.502
Verified: True
